# Credit Scoring Tool - Notebook de Ejemplo

Esta notebook demuestra todas las funcionalidades de la paquetería **credit-scoring-tool** usando el dataset `application_train.csv`.

**Secciones:**
1. Carga de Datos
2. Preprocesamiento (Missing Values, Encoding, Binning, WOE)
3. Selección de Variables (IV, Correlación)
4. Modelos (Logistic Regression, Random Forest, Neural Network, XGBoost)
5. Evaluación (Métricas de Clasificación y Credit Scoring)
6. Visualización
7. Pipeline Integrado

---
## 1. Carga de Datos

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split

# Cargamos y preparamos datos
datos = pd.read_csv('application_train.csv')

X = datos.drop(columns=['TARGET', 'SK_ID_CURR'])
y = datos['TARGET']

print(f"Dataset shape: {X.shape}")
print(f"Target distribution:
{y.value_counts(normalize=True)}")
X.head()

In [ ]:
# División en train/test
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")

---
## 2. Preprocesamiento

### 2.1 Manejo de Valores Faltantes

In [ ]:
from creditScoring.preprocessing import (
    handle_missing_values,
    handle_numeric_missing_values,
    handle_categorical_missing_values,
)

# Ver valores faltantes antes
print("Valores faltantes por columna (top 10):" )
print(X_train.isnull().sum().sort_values(ascending=False).head(10))

In [ ]:
# Manejo de numéricos con mediana
X_num_clean = handle_numeric_missing_values(X_train, strategy="median")
print(f"
Valores faltantes numéricos después de mediana: {X_num_clean.select_dtypes(include=['number']).isnull().sum().sum()}")

# Manejo de categóricos con moda
X_cat_clean = handle_categorical_missing_values(X_train, strategy="mode")
print(f"Valores faltantes categóricos después de moda: {X_cat_clean.select_dtypes(exclude=['number']).isnull().sum().sum()}")

In [ ]:
# Función combinada
X_train_clean = handle_missing_values(
    X_train,
    numeric_strategy="median",
    categorical_strategy="mode",
)
X_test_clean = handle_missing_values(
    X_test,
    numeric_strategy="median",
    categorical_strategy="mode",
)
print(f"Total missing después de limpieza (train): {X_train_clean.isnull().sum().sum()}")
print(f"Total missing después de limpieza (test): {X_test_clean.isnull().sum().sum()}")

### 2.2 Encoding Categórico

In [ ]:
from creditScoring.preprocessing import CategoricalEncoder

# Encoding por frecuencia
encoder = CategoricalEncoder(method="frequency")
encoder.fit(X_train_clean, y_train)
X_train_encoded = encoder.transform(X_train_clean)
X_test_encoded = encoder.transform(X_test_clean)

print(f"Columnas categóricas codificadas: {len(encoder.columns_)}")
print(f"Shape después de encoding: {X_train_encoded.shape}")
X_train_encoded.head()

### 2.3 Binning / Discretización

In [ ]:
from creditScoring.preprocessing import BinningTransformer

# Binning monótono (requiere target)
binner = BinningTransformer(method="monotonic", n_bins=5, min_bin_pct=0.05)
binner.fit(X_train_encoded, y_train)
X_train_binned = binner.transform(X_train_encoded)

print(f"Columnas numéricas binneadas: {len(binner.numeric_columns_)}")
print(f"
Ejemplo de bins para la primera columna numérica:")
col_ejemplo = binner.numeric_columns_[0]
print(f"  {col_ejemplo}: {X_train_binned[col_ejemplo].value_counts().sort_index()}")

### 2.4 WOE (Weight of Evidence)

In [ ]:
from creditScoring.preprocessing import WOETransformer, calculate_woe_iv, calculate_iv_for_dataframe

# Calcular WOE/IV para una variable individual
X_train_str = X_train_binned.astype(str)
ejemplo_col = X_train_str.columns[0]
woe_table, iv_value = calculate_woe_iv(X_train_str[ejemplo_col], y_train)
print(f"WOE table para '{ejemplo_col}' (IV={iv_value:.4f}):")
print(woe_table[['bin', 'woe', 'iv_component']].to_string(index=False))

In [ ]:
# Transformador WOE completo
woe_transformer = WOETransformer()
woe_transformer.fit(X_train_str, y_train)
X_train_woe = woe_transformer.transform(X_train_str)

print(f"
Top 10 variables por IV:")
print(woe_transformer.iv_table_.head(10).to_string(index=False))

---
## 3. Selección de Variables

### 3.1 Selección por Information Value (IV)

In [ ]:
from creditScoring.feature_selection import IVFeatureSelector

# Selección basada en IV mínimo
iv_selector = IVFeatureSelector(min_iv=0.02)
iv_selector.fit(X_train_woe, y_train)

print(f"Variables seleccionadas (IV >= 0.02): {len(iv_selector.selected_features_)}")
print(f"Variables descartadas: {X_train_woe.shape[1] - len(iv_selector.selected_features_)}")
print(f"
Ranking IV (top 15):")
print(iv_selector.get_ranking().head(15).to_string(index=False))

### 3.2 Eliminación por Correlación

In [ ]:
from creditScoring.feature_selection import correlation_matrix, remove_correlated_features

# Matriz de correlación
X_selected = iv_selector.transform(X_train_woe)
corr = correlation_matrix(X_selected, method="pearson")
print(f"Shape matriz de correlación: {corr.shape}")

# Eliminar features altamente correlacionadas
ranking = iv_selector.get_ranking()
final_features = remove_correlated_features(
    X_selected, ranking, threshold=0.8, method="pearson"
)
print(f"
Features después de filtro de correlación (threshold=0.8): {len(final_features)}")
print(f"Features eliminadas por correlación: {len(iv_selector.selected_features_) - len(final_features)}")

---
## 4. Modelos

Probamos los 4 algoritmos disponibles en la paquetería.

### 4.1 Regresión Logística

In [ ]:
from creditScoring.models import LogisticRegressionModel

# Preparar datos finales para modelos
X_train_final = X_train_woe[final_features]
X_test_str = X_test_encoded.astype(str)
X_test_woe = woe_transformer.transform(X_test_str)
X_test_final = X_test_woe[final_features]

# Entrenar Logistic Regression
lr_model = LogisticRegressionModel(random_state=42)
lr_model.fit(X_train_final, y_train)

# Predicciones
y_pred_lr = lr_model.predict(X_test_final)
y_prob_lr = lr_model.predict_proba(X_test_final)

# Coeficientes
coefs = lr_model.get_coefficients()
print("Top 10 coeficientes (valor absoluto):")
print(coefs.abs().sort_values(ascending=False).head(10))

### 4.2 Random Forest

In [ ]:
from creditScoring.models import RandomForestModel

rf_model = RandomForestModel(n_estimators=100, random_state=42)
rf_model.fit(X_train_final, y_train)

y_pred_rf = rf_model.predict(X_test_final)
y_prob_rf = rf_model.predict_proba(X_test_final)

# Feature importance
importances = rf_model.feature_importance()
print("Top 10 features por importancia:")
print(importances.sort_values(ascending=False).head(10))

### 4.3 Red Neuronal (MLP)

In [ ]:
from creditScoring.models import NeuralNetworkModel

nn_model = NeuralNetworkModel(
    hidden_layer_sizes=(64, 32),
    max_iter=300,
    random_state=42,
)
nn_model.fit(X_train_final, y_train)

y_pred_nn = nn_model.predict(X_test_final)
y_prob_nn = nn_model.predict_proba(X_test_final)

print(f"Predicciones generadas: {len(y_pred_nn)}")
print(f"Rango de probabilidades: [{y_prob_nn.min():.4f}, {y_prob_nn.max():.4f}]")

### 4.4 XGBoost

In [ ]:
from creditScoring.models import XGBoostModel

xgb_model = XGBoostModel(
    n_estimators=100,
    learning_rate=0.05,
    max_depth=4,
    random_state=42,
)
xgb_model.fit(X_train_final, y_train)

y_pred_xgb = xgb_model.predict(X_test_final)
y_prob_xgb = xgb_model.predict_proba(X_test_final)

print(f"Predicciones generadas: {len(y_pred_xgb)}")
print(f"Rango de probabilidades: [{y_prob_xgb.min():.4f}, {y_prob_xgb.max():.4f}]")

---
## 5. Evaluación

### 5.1 Métricas de Clasificación

In [ ]:
from creditScoring.evaluation import classification_report_dict

modelos = {
    "Logistic Regression": y_prob_lr,
    "Random Forest": y_prob_rf,
    "Neural Network": y_prob_nn,
    "XGBoost": y_prob_xgb,
}

resultados = []
for nombre, probs in modelos.items():
    metrics = classification_report_dict(y_test, probs, threshold=0.5)
    resultados.append({
        "Modelo": nombre,
        "Accuracy": metrics["accuracy"],
        "Precision": metrics["precision"],
        "Recall": metrics["recall"],
        "F1": metrics["f1"],
        "ROC AUC": metrics["roc_auc"],
        "PR AUC": metrics["pr_auc"],
    })

df_resultados = pd.DataFrame(resultados)
print(df_resultados.to_string(index=False))

### 5.2 Métricas de Credit Scoring

In [ ]:
from creditScoring.evaluation import (
    ks_statistic,
    gini_coefficient,
    population_stability_index,
    lift_gain_table,
    divergence_index,
)

credit_results = []
for nombre, probs in modelos.items():
    ks = ks_statistic(y_test, probs)
    gini = gini_coefficient(y_test, probs)
    credit_results.append({
        "Modelo": nombre,
        "KS Statistic": ks,
        "Gini": gini,
    })

df_credit = pd.DataFrame(credit_results)
print("Métricas de Credit Scoring:")
print(df_credit.to_string(index=False))

In [ ]:
# Tabla de Lift/Gain para el mejor modelo
print("
Tabla de Lift/Gain (XGBoost):")
lift_table = lift_gain_table(y_test, y_prob_xgb, n_bins=10)
print(lift_table.to_string(index=False))

In [ ]:
# Population Stability Index (comparando train vs test)
y_prob_train_xgb = xgb_model.predict_proba(X_train_final)
psi = population_stability_index(
    pd.Series(y_prob_train_xgb),
    pd.Series(y_prob_xgb),
    bins=10,
)
print(f"
PSI (Train vs Test): {psi:.6f}")
if psi < 0.1:
    print("  → Población estable (PSI < 0.1)")
elif psi < 0.2:
    print("  → Cambio moderado (0.1 <= PSI < 0.2)")
else:
    print("  → Cambio significativo (PSI >= 0.2)")

In [ ]:
# Divergence Index
good_scores = pd.Series(y_prob_xgb[y_test == 0])
bad_scores = pd.Series(y_prob_xgb[y_test == 1])
div_idx = divergence_index(good_scores, bad_scores)
print(f"Divergence Index: {div_idx:.4f}")

### 5.3 ModelResults (Contenedor de Resultados)

In [ ]:
from creditScoring.evaluation import ModelResults

# Usar el contenedor integrado
results = ModelResults(
    classification_metrics=classification_report_dict(y_test, y_prob_xgb),
    credit_metrics={"ks": ks_statistic(y_test, y_prob_xgb), "gini": gini_coefficient(y_test, y_prob_xgb)},
    metadata={"model": "XGBoost", "threshold": 0.5},
)

print(results.summary())
print("
Resultados como DataFrame:")
results.to_dataframe()

---
## 6. Visualización

In [ ]:
from creditScoring.visualization import (
    plot_distributions,
    plot_correlation_heatmap,
    plot_score_distribution,
    plot_roc_curve,
    plot_ks_curve,
    plot_feature_importance,
    plot_lift_gain,
)
import matplotlib
matplotlib.use("Agg")  # Para ambientes sin display
import matplotlib.pyplot as plt

### 6.1 Distribuciones de Variables

In [ ]:
# Distribución de algunas variables numéricas
num_cols = X_train.select_dtypes(include=["number"]).columns[:4].tolist()
fig, axes = plot_distributions(X_train, num_cols)
plt.tight_layout()
plt.show()

### 6.2 Heatmap de Correlación

In [ ]:
fig, ax = plot_correlation_heatmap(X_train_final.iloc[:, :15])
plt.tight_layout()
plt.show()

### 6.3 Distribución de Scores

In [ ]:
fig, ax = plot_score_distribution(pd.Series(y_prob_xgb), y_test)
plt.show()

### 6.4 Curva ROC

In [ ]:
fig, ax = plot_roc_curve(y_test, y_prob_xgb)
plt.show()

### 6.5 Curva KS

In [ ]:
fig, ax = plot_ks_curve(y_test, y_prob_xgb)
plt.show()

### 6.6 Importancia de Features

In [ ]:
fig, ax = plot_feature_importance(importances.sort_values(ascending=False).head(15))
plt.tight_layout()
plt.show()

### 6.7 Lift y Gain

In [ ]:
fig, axes = plot_lift_gain(y_test, y_prob_xgb)
plt.tight_layout()
plt.show()

---
## 7. Pipeline Integrado

La clase  integra todo el flujo de preprocesamiento, selección de variables, entrenamiento y evaluación.

### 7.1 Pipeline con Regresión Logística (default)

In [ ]:
from creditScoring import CreditScoringPipeline, PipelineConfig, get_default_config

# Configuración por defecto (logistic regression)
config = get_default_config()
print("Configuración por defecto:")
for key, value in config.items():
    print(f"  {key}: {value}")

In [ ]:
# Crear y entrenar pipeline
pipeline_lr = CreditScoringPipeline(config)
pipeline_lr.fit(X_train, y_train)

# Evaluar
results_lr = pipeline_lr.evaluate(X_test, y_test)
print("Pipeline Logistic Regression:")
print(results_lr.summary())

### 7.2 Pipeline con Random Forest

In [ ]:
config_rf = PipelineConfig(model_type="random_forest")
pipeline_rf = CreditScoringPipeline(config_rf)
pipeline_rf.fit(X_train, y_train)

results_rf = pipeline_rf.evaluate(X_test, y_test)
print("Pipeline Random Forest:")
print(results_rf.summary())

### 7.3 Pipeline con Neural Network

In [ ]:
config_nn = PipelineConfig(model_type="neural_network")
pipeline_nn = CreditScoringPipeline(config_nn)
pipeline_nn.fit(X_train, y_train)

results_nn = pipeline_nn.evaluate(X_test, y_test)
print("Pipeline Neural Network:")
print(results_nn.summary())

### 7.4 Pipeline con XGBoost

In [ ]:
config_xgb = PipelineConfig(model_type="xgboost")
pipeline_xgb = CreditScoringPipeline(config_xgb)
pipeline_xgb.fit(X_train, y_train)

results_xgb = pipeline_xgb.evaluate(X_test, y_test)
print("Pipeline XGBoost:")
print(results_xgb.summary())

### 7.5 Comparación Final de Pipelines

In [ ]:
comparacion = pd.DataFrame([
    {"Pipeline": "Logistic Regression", **results_lr.credit_metrics},
    {"Pipeline": "Random Forest", **results_rf.credit_metrics},
    {"Pipeline": "Neural Network", **results_nn.credit_metrics},
    {"Pipeline": "XGBoost", **results_xgb.credit_metrics},
])
print("Comparación de Pipelines (Métricas Credit Scoring):")
print(comparacion.to_string(index=False))

### 7.6 Guardar y Cargar Pipeline

In [ ]:
# Guardar el mejor pipeline
pipeline_xgb.save("pipeline_xgboost.joblib")
print("Pipeline guardado en 'pipeline_xgboost.joblib'")

# Cargar pipeline
pipeline_loaded = CreditScoringPipeline.load("pipeline_xgboost.joblib")
results_loaded = pipeline_loaded.evaluate(X_test, y_test)
print("
Pipeline cargado - verificación:")
print(results_loaded.summary())

---
## Resumen

Esta notebook demostró las siguientes funcionalidades de **credit-scoring-tool**:

| Módulo | Funcionalidades |
|--------|----------------|
|  | Missing values, Encoding categórico, Binning, WOE/IV |
|  | Selección por IV, Filtro de correlación |
|  | Logistic Regression, Random Forest, Neural Network, XGBoost |
|  | Clasificación (accuracy, precision, recall, F1, AUC), Credit Scoring (KS, Gini, PSI, Lift, Divergence) |
|  | Distribuciones, Correlación, Scores, ROC, KS, Feature Importance, Lift/Gain |
|  | Pipeline integrado end-to-end con save/load |